<a href="https://colab.research.google.com/github/seshankbagavath/Multi-Factor-Equity-Screener/blob/main/screener_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# ════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP & DEPENDENCIES
# ════════════════════════════════════════════════════════════════════════
# Installs all required libraries. Run once per Colab session.

!pip install -q yfinance pandas numpy requests matplotlib plotly scipy tqdm

import warnings
warnings.filterwarnings("ignore")

import os
import time
import json
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from scipy import stats
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("Environment ready.")

Environment ready.


In [13]:
# ════════════════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION
# ════════════════════════════════════════════════════════════════════════

class Config:
    # --- Universe -------------------------------------------------------
    UNIVERSE = [
        "AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "TSLA", "JPM",
        "V", "JNJ", "WMT", "PG", "MA", "HD", "CVX", "ABBV", "PEP", "KO",
        "COST", "MRK", "ADBE", "CSCO", "CRM", "MCD", "NKE", "INTC", "AMD",
        "TXN", "QCOM", "ORCL", "IBM", "GE", "CAT", "BA", "GS", "MS", "XOM",
        "PFE", "T", "VZ", "DIS", "NFLX", "PYPL", "SBUX", "LOW", "UNH",
        "HON", "UPS", "DE", "LMT",
    ]

    # --- Price lookbacks (calendar) ------------------------------------
    PRICE_PERIOD = "2y"
    MOMENTUM_LOOKBACK = 252
    MOMENTUM_SKIP = 21
    VOL_LOOKBACK = 252

    # --- Factor weights ------------------------------------------------
    FACTOR_WEIGHTS = {
        "value":         0.25,
        "profitability": 0.25,
        "momentum":      0.20,
        "volatility":    0.15,
        "growth":        0.15,
    }

    # --- Cleaning ------------------------------------------------------
    WINSOR_LIMITS = (0.01, 0.99)
    MIN_FACTOR_COVERAGE = 0.50

    # --- Optional API enrichment ---------------------------------------
    FMP_API_KEY = ""
    SEC_USER_AGENT = "screener-project contact@example.com"

    # --- Caching --------------------------------------------------------
    CACHE_DIR = "data/raw"


cfg = Config()
os.makedirs(cfg.CACHE_DIR, exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# --- Override with full S&P 500 universe ---
try:
    import io
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    html = requests.get(url, headers=headers, timeout=15).text
    sp500 = pd.read_html(io.StringIO(html))[0]["Symbol"] \
        .str.replace(".", "-", regex=False).tolist()
    cfg.UNIVERSE = sp500
    print(f"Loaded {len(cfg.UNIVERSE)} S&P 500 tickers.")
except Exception as e:
    print(f"Wikipedia fetch failed ({e}); using built-in 50-ticker list.")

print(f"Configured {len(cfg.UNIVERSE)} tickers across "
      f"{len(cfg.FACTOR_WEIGHTS)} factor families.")

Loaded 503 S&P 500 tickers.
Configured 503 tickers across 5 factor families.


In [14]:
import glob, os
for f in glob.glob("data/raw/*"):
    os.remove(f)
print("Cache cleared.")

Cache cleared.


In [15]:
# ════════════════════════════════════════════════════════════════════════
# CELL 3 — DATA COLLECTION PIPELINE
# ════════════════════════════════════════════════════════════════════════
# Pulls prices (batched) and per-ticker fundamentals. Every network call is
# wrapped in try/except so a single bad ticker never aborts the run. Raw
# fundamentals are cached to disk to avoid re-hitting the API on re-runs.

def download_prices(tickers, period, cache_dir):
    """Batch-download adjusted close prices. Returns a wide DataFrame
    indexed by date with one column per ticker."""
    cache_path = os.path.join(cache_dir, f"prices_{period}.pkl")
    if os.path.exists(cache_path):
        age_h = (time.time() - os.path.getmtime(cache_path)) / 3600
        if age_h < 24:  # reuse cache for a day
            print("Loading cached prices.")
            return pd.read_pickle(cache_path)

    print("Downloading prices...")
    try:
        raw = yf.download(
            tickers, period=period, auto_adjust=True,
            progress=False, threads=True,
        )
        # Handle single- vs multi-ticker column shapes
        prices = raw["Close"] if "Close" in raw else raw
        if isinstance(prices, pd.Series):
            prices = prices.to_frame()
        prices = prices.dropna(how="all").ffill()
        prices.to_pickle(cache_path)
        return prices
    except Exception as e:
        print(f"Price download failed: {e}")
        return pd.DataFrame()


def _safe_get(d, key, default=np.nan):
    """Defensive dict access for yfinance .info payloads."""
    try:
        v = d.get(key, default)
        return v if v is not None else default
    except Exception:
        return default


def download_fundamentals(tickers, cache_dir):
    """Pull per-ticker fundamental snapshot from yfinance. Cached as JSON.
    Each ticker isolated in try/except for fault tolerance."""
    cache_path = os.path.join(cache_dir, "fundamentals.json")
    if os.path.exists(cache_path):
        age_h = (time.time() - os.path.getmtime(cache_path)) / 3600
        if age_h < 24:
            print("Loading cached fundamentals.")
            with open(cache_path) as f:
                return pd.DataFrame(json.load(f)).T

    records = {}
    for tk in tqdm(tickers, desc="Fundamentals"):
        try:
            info = yf.Ticker(tk).info
            records[tk] = {
                "sector":        _safe_get(info, "sector", "Unknown"),
                "marketCap":     _safe_get(info, "marketCap"),
                "trailingPE":    _safe_get(info, "trailingPE"),
                "priceToBook":   _safe_get(info, "priceToBook"),
                "enterpriseToEbitda": _safe_get(info, "enterpriseToEbitda"),
                "returnOnEquity":     _safe_get(info, "returnOnEquity"),
                "returnOnAssets":     _safe_get(info, "returnOnAssets"),
                "grossMargins":       _safe_get(info, "grossMargins"),
                "profitMargins":      _safe_get(info, "profitMargins"),
                "revenueGrowth":      _safe_get(info, "revenueGrowth"),
                "earningsGrowth":     _safe_get(info, "earningsGrowth"),
                "freeCashflow":       _safe_get(info, "freeCashflow"),
            }
            time.sleep(0.05)  # be polite to the endpoint
        except Exception as e:
            print(f"  skip {tk}: {e}")
            continue

    with open(cache_path, "w") as f:
        json.dump(records, f)
    return pd.DataFrame(records).T


# --- Run the pipeline ---------------------------------------------------
prices = download_prices(cfg.UNIVERSE, cfg.PRICE_PERIOD, cfg.CACHE_DIR)
fundamentals = download_fundamentals(cfg.UNIVERSE, cfg.CACHE_DIR)

print(f"\nPrices: {prices.shape[0]} days × {prices.shape[1]} tickers")
print(f"Fundamentals: {fundamentals.shape[0]} tickers × "
      f"{fundamentals.shape[1]} fields")

Fundamentals:   0%|          | 0/503 [00:00<?, ?it/s]


Prices: 502 days × 503 tickers
Fundamentals: 503 tickers × 12 fields


In [16]:
# ════════════════════════════════════════════════════════════════════════
# CELL 4 — OPTIONAL: SEC EDGAR & FMP ENRICHMENT (graceful fallback)
# ════════════════════════════════════════════════════════════════════════
# Demonstrates authoritative-source integration. Both functions degrade
# silently to NaN if unavailable, so the screener still runs end-to-end.

def fetch_sec_concept(cik, tag="Revenues", user_agent=cfg.SEC_USER_AGENT):
    """Fetch a single US-GAAP concept from SEC EDGAR company-facts API.
    CIK must be zero-padded to 10 digits. Returns latest annual value."""
    try:
        cik_padded = str(cik).zfill(10)
        url = (f"https://data.sec.gov/api/xbrl/companyconcept/"
               f"CIK{cik_padded}/us-gaap/{tag}.json")
        r = requests.get(url, headers={"User-Agent": user_agent}, timeout=10)
        r.raise_for_status()
        units = r.json()["units"]["USD"]
        annual = [u for u in units if u.get("form") == "10-K"]
        return annual[-1]["val"] if annual else np.nan
    except Exception:
        return np.nan


def fetch_fmp_ratios(ticker, api_key=cfg.FMP_API_KEY):
    """Fetch TTM ratios from Financial Modeling Prep. No-op without a key."""
    if not api_key:
        return {}
    try:
        url = (f"https://financialmodelingprep.com/api/v3/"
               f"ratios-ttm/{ticker}?apikey={api_key}")
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        return data[0] if data else {}
    except Exception:
        return {}

print("EDGAR / FMP helpers ready (enrichment optional).")

EDGAR / FMP helpers ready (enrichment optional).


In [17]:
# ════════════════════════════════════════════════════════════════════════
# CELL 5 — DATA CLEANING
# ════════════════════════════════════════════════════════════════════════
# Coerce types, neutralize infinities, winsorize outliers, and impute
# missing values using the SECTOR MEDIAN (more robust than global mean).

def coerce_numeric(df, non_numeric=("sector",)):
    """Force all columns except listed ones to numeric, NaN on failure."""
    out = df.copy()
    for c in out.columns:
        if c not in non_numeric:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    out = out.replace([np.inf, -np.inf], np.nan)
    return out


def winsorize_series(s, limits):
    """Clip a numeric series at the given lower/upper quantiles."""
    lo, hi = s.quantile(limits[0]), s.quantile(limits[1])
    return s.clip(lower=lo, upper=hi)


def clean_fundamentals(df, cfg):
    """Full cleaning routine: numeric coercion, winsorization,
    sector-median imputation."""
    df = coerce_numeric(df)
    numeric_cols = [c for c in df.columns if c != "sector"]

    # Winsorize each metric to tame outliers
    for c in numeric_cols:
        df[c] = winsorize_series(df[c], cfg.WINSOR_LIMITS)

    # Impute missing values with the within-sector median, then global median
    for c in numeric_cols:
        df[c] = df.groupby("sector")[c].transform(
            lambda x: x.fillna(x.median())
        )
        df[c] = df[c].fillna(df[c].median())

    return df


fund_clean = clean_fundamentals(fundamentals, cfg)
print("Cleaned fundamentals. Missing values remaining:",
      int(fund_clean.drop(columns=["sector"]).isna().sum().sum()))
fund_clean.head()

Cleaned fundamentals. Missing values remaining: 0


,sector,marketCap,trailingPE,priceToBook,enterpriseToEbitda,returnOnEquity,returnOnAssets,grossMargins,profitMargins,revenueGrowth,earningsGrowth,freeCashflow
MMM,Industrials,"87,487,692,800.000",32.258,26.813,14.706,0.715,0.081,0.397,0.111,0.013,-0.397,"2,322,374,912.000"
AOS,Industrials,"8,488,221,071.360",16.153,4.446,10.524,0.283,0.128,0.388,0.138,-0.019,-0.105,"495,612,512.000"
ABT,Healthcare,"159,288,803,328.000",25.616,3.060,15.767,0.123,0.056,0.565,0.139,0.078,-0.197,"6,341,125,120.000"
ABBV,Healthcare,"412,643,295,232.000",114.488,-62.000,15.990,0.142,0.100,0.720,0.058,0.124,-0.462,"20,811,624,448.000"
ACN,Technology,"78,145,003,520.000",10.200,2.450,5.992,0.244,0.109,0.320,0.107,0.056,0.090,"12,089,380,864.000"


In [33]:
# ════════════════════════════════════════════════════════════════════════
# CELL 6 — FEATURE ENGINEERING: FACTOR CALCULATION
# ════════════════════════════════════════════════════════════════════════
# Build raw metrics for the five factor families. Price-based factors
# (momentum, volatility) come from the price panel; the rest from
# fundamentals. Each metric is oriented so HIGHER = MORE ATTRACTIVE later.
#
# CLEANING PHILOSOPHY: every DERIVED or INVERTED metric is winsorized at the
# point it is created. Cell 5 winsorized the raw fundamentals, but inverted
# valuation yields (1/PE, 1/PB, ...) and price factors (momentum, vol) are
# built HERE, so a near-zero denominator (e.g. P/B ~ 0 -> book_yield = 576)
# or a spinoff's discontinuous price history would otherwise leak through.

def compute_price_factors(prices, cfg):
    """Momentum (12-1) and inverse realized volatility from price history.
    Tickers without a full lookback window are nulled out so partial
    histories (recent spinoffs/IPOs) don't inject extreme outliers."""
    rets = prices.pct_change()

    # 12-1 momentum: return over lookback, skipping the most recent month
    end = prices.iloc[-cfg.MOMENTUM_SKIP - 1]
    start = prices.iloc[-cfg.MOMENTUM_LOOKBACK - 1]
    momentum = (end / start) - 1.0

    # Annualized realized volatility (we will invert in normalization)
    vol = rets.iloc[-cfg.VOL_LOOKBACK:].std() * np.sqrt(252)

    # Require a full lookback of real history; otherwise null the factors
    valid_history = prices.notna().sum() >= cfg.MOMENTUM_LOOKBACK
    momentum = momentum.where(valid_history)
    vol = vol.where(valid_history)

    return pd.DataFrame({"momentum_raw": momentum, "volatility_raw": vol})


def build_factor_table(fund, price_factors):
    """Assemble one row per ticker with raw factor inputs.
    Valuation metrics are inverted (1/x) so higher = cheaper = better."""
    f = pd.DataFrame(index=fund.index)
    f["sector"] = fund["sector"]

    # --- VALUE (invert: cheaper is better) -----------------------------
    f["earnings_yield"] = 1.0 / fund["trailingPE"]
    f["book_yield"]     = 1.0 / fund["priceToBook"]
    f["ebitda_yield"]   = 1.0 / fund["enterpriseToEbitda"]

    # --- PROFITABILITY (higher is better, already oriented) ------------
    f["roe"]          = fund["returnOnEquity"]
    f["roa"]          = fund["returnOnAssets"]
    f["gross_margin"] = fund["grossMargins"]
    f["net_margin"]   = fund["profitMargins"]

    # --- GROWTH --------------------------------------------------------
    f["revenue_growth"]  = fund["revenueGrowth"]
    f["earnings_growth"] = fund["earningsGrowth"]

    # --- MOMENTUM & VOLATILITY (from prices) ---------------------------
    f = f.join(price_factors)
    f["momentum"]   = f["momentum_raw"]
    # Invert volatility so a LOW-vol stock scores HIGH (low-vol tilt)
    f["inv_volatility"] = -f["volatility_raw"]

    # --- Winsorize price-derived factors to kill spinoff/IPO artifacts -
    for col in ["momentum", "inv_volatility"]:
        lo, hi = f[col].quantile(0.02), f[col].quantile(0.98)
        f[col] = f[col].clip(lower=lo, upper=hi)

    # --- Winsorize inverted valuation yields ---------------------------
    # A near-zero P/B or P/E produces an absurd yield (e.g. book_yield = 576)
    # that survives the Cell 5 clip because it's created here, post-inversion.
    for col in ["earnings_yield", "book_yield", "ebitda_yield"]:
        lo, hi = f[col].quantile(0.02), f[col].quantile(0.98)
        f[col] = f[col].clip(lower=lo, upper=hi)

    return f


price_factors = compute_price_factors(prices, cfg)
factor_table = build_factor_table(fund_clean, price_factors)

# Map each raw metric to its factor family for later aggregation
FACTOR_MAP = {
    "value":         ["earnings_yield", "book_yield", "ebitda_yield"],
    "profitability": ["roe", "roa", "gross_margin", "net_margin"],
    "momentum":      ["momentum"],
    "volatility":    ["inv_volatility"],
    "growth":        ["revenue_growth", "earnings_growth"],
}

print("Factor table built:", factor_table.shape)
print("Momentum range:  "
      f"{factor_table['momentum'].min():.2f} to "
      f"{factor_table['momentum'].max():.2f}")
print("Book yield range: "
      f"{factor_table['book_yield'].min():.3f} to "
      f"{factor_table['book_yield'].max():.3f}")
factor_table.head()

Factor table built: (503, 14)
Momentum range:  -0.48 to 2.78
Book yield range: -0.072 to 1.066


,sector,earnings_yield,book_yield,ebitda_yield,roe,roa,gross_margin,net_margin,revenue_growth,earnings_growth,momentum_raw,volatility_raw,momentum,inv_volatility
MMM,Industrials,0.031,0.037,0.068,0.715,0.081,0.397,0.111,0.013,-0.397,0.052,0.260,0.052,-0.260
AOS,Industrials,0.062,0.225,0.095,0.283,0.128,0.388,0.138,-0.019,-0.105,-0.088,0.260,-0.088,-0.260
ABT,Healthcare,0.039,0.327,0.063,0.123,0.056,0.565,0.139,0.078,-0.197,-0.331,0.249,-0.331,-0.249
ABBV,Healthcare,0.009,-0.016,0.063,0.142,0.100,0.720,0.058,0.124,-0.462,0.212,0.252,0.212,-0.252
ACN,Technology,0.098,0.408,0.167,0.244,0.109,0.320,0.107,0.056,0.090,-0.384,0.400,-0.384,-0.400


In [34]:
# ════════════════════════════════════════════════════════════════════════
# CELL 7 — NORMALIZATION & COMPOSITE SCORING
# ════════════════════════════════════════════════════════════════════════
# Cross-sectional z-score each metric, average within a factor family to get
# a family score, then take a weighted sum across families for the composite.
# Weights are renormalized over only the families a stock actually has data
# for, so partial coverage never unfairly penalizes a name.

def zscore(s):
    """Robust cross-sectional z-score; returns 0s if no dispersion."""
    mu, sigma = s.mean(), s.std()
    if sigma == 0 or np.isnan(sigma):
        return pd.Series(0.0, index=s.index)
    return (s - mu) / sigma


def compute_family_scores(factor_table, factor_map):
    """Z-score every metric, then average metrics within each family.
    NaNs (e.g. short-history names with nulled momentum) are filled to the
    cross-sectional median BEFORE z-scoring so they land at a neutral 0."""
    fam_scores = pd.DataFrame(index=factor_table.index)
    for family, metrics in factor_map.items():
        present = [m for m in metrics if m in factor_table.columns]
        if not present:
            continue
        block = factor_table[present].apply(
            lambda col: col.fillna(col.median())
        )
        z = block.apply(zscore)
        fam_scores[family] = z.mean(axis=1)
    return fam_scores


def composite_score(fam_scores, weights):
    """Weighted sum of family z-scores with per-row weight renormalization."""
    w = pd.Series(weights)
    w = w / w.sum()

    scores, valid_cols = [], [c for c in fam_scores.columns if c in w.index]
    for _, row in fam_scores[valid_cols].iterrows():
        mask = row.notna()
        if mask.sum() == 0:
            scores.append(np.nan)
            continue
        rw = w[valid_cols][mask] / w[valid_cols][mask].sum()
        scores.append((row[mask] * rw).sum())
    return pd.Series(scores, index=fam_scores.index)


family_scores = compute_family_scores(factor_table, FACTOR_MAP)
family_scores["composite"] = composite_score(family_scores, cfg.FACTOR_WEIGHTS)

# Rank and bucket into deciles (decile 1 = best)
results = family_scores.copy()
results["sector"] = factor_table["sector"]
results = results.sort_values("composite", ascending=False)
results["rank"] = range(1, len(results) + 1)
results["decile"] = pd.qcut(
    results["composite"].rank(method="first", ascending=False),
    q=min(10, len(results)), labels=False
) + 1

results.to_csv("data/processed/screener_results.csv")
print("Top 10 names:")
results.head(10)[["composite", "rank", "decile", "sector"]]

Top 10 names:


,composite,rank,decile,sector
MU,1.093,1,1,Technology
AFL,1.020,2,1,Financial Services
STX,0.972,3,1,Technology
EQT,0.955,4,1,Energy
NVDA,0.894,5,1,Technology
FIS,0.883,6,1,Technology
WDC,0.840,7,1,Technology
SNDK,0.819,8,1,Technology
VICI,0.814,9,1,Real Estate
APA,0.790,10,1,Energy


In [32]:
print(factor_table.loc["FDXF"])

sector             Industrials
earnings_yield           0.027
book_yield             576.019
ebitda_yield             0.295
roe                      0.233
roa                      0.071
gross_margin             0.297
net_margin               0.104
revenue_growth          -0.047
earnings_growth          0.120
momentum_raw               NaN
volatility_raw             NaN
momentum                   NaN
inv_volatility             NaN
Name: FDXF, dtype: object


In [27]:
# Inspect the factor fingerprint of your top name
print(results.loc["SNDK", fam_cols])
print("\nRaw inputs:")
print(factor_table.loc["SNDK"])

value           -0.803
profitability    1.294
momentum        18.765
volatility      -4.963
growth           2.212
Name: SNDK, dtype: object

Raw inputs:
sector             Technology
earnings_yield          0.015
book_yield              0.048
ebitda_yield            0.020
roe                     0.393
roa                     0.228
gross_margin            0.560
net_margin              0.342
revenue_growth          0.914
earnings_growth         0.245
momentum_raw           30.495
volatility_raw          1.015
momentum               30.495
inv_volatility         -1.015
Name: SNDK, dtype: object


In [20]:
# ════════════════════════════════════════════════════════════════════════
# CELL 8 — VISUALIZATION 1: TOP RANKINGS (interactive bar)
# ════════════════════════════════════════════════════════════════════════

def plot_rankings(results, top_n=15):
    top = results.head(top_n).iloc[::-1]  # reverse for horizontal bar
    fig = px.bar(
        top, x="composite", y=top.index, orientation="h",
        color="composite", color_continuous_scale="Tealgrn",
        title=f"Top {top_n} Stocks by Composite Factor Score",
        labels={"composite": "Composite Z-Score", "index": "Ticker"},
    )
    fig.update_layout(template="plotly_white", height=520,
                      coloraxis_showscale=False,
                      font=dict(family="Space Grotesk, sans-serif"))
    return fig

plot_rankings(results).show()

In [21]:
# ════════════════════════════════════════════════════════════════════════
# CELL 9 — VISUALIZATION 2: FACTOR HEATMAP
# ════════════════════════════════════════════════════════════════════════
# Z-score fingerprint of the top names across the five factor families.

def plot_heatmap(results, family_cols, top_n=20):
    top = results.head(top_n)
    z = top[family_cols]
    fig = go.Figure(go.Heatmap(
        z=z.values, x=family_cols, y=top.index,
        colorscale="RdYlGn", zmid=0,
        colorbar=dict(title="Z"),
    ))
    fig.update_layout(
        title=f"Factor Z-Score Heatmap — Top {top_n}",
        template="plotly_white", height=640,
        font=dict(family="Space Grotesk, sans-serif"),
    )
    return fig

fam_cols = list(cfg.FACTOR_WEIGHTS.keys())
plot_heatmap(results, fam_cols).show()

In [22]:
# ════════════════════════════════════════════════════════════════════════
# CELL 10 — VISUALIZATION 3: VALUE vs PROFITABILITY SCATTER
# ════════════════════════════════════════════════════════════════════════
# Classic "quality vs cheapness" map. Bubble size = composite score;
# names in the upper-right are cheap AND profitable.

def plot_value_quality(results):
    df = results.dropna(subset=["value", "profitability", "composite"]).copy()
    df["size"] = (df["composite"] - df["composite"].min() + 0.5)
    fig = px.scatter(
        df, x="value", y="profitability",
        size="size", color="sector", text=df.index,
        title="Value vs Profitability (bubble = composite score)",
        labels={"value": "Value Z-Score (cheap →)",
                "profitability": "Profitability Z-Score (quality →)"},
    )
    fig.update_traces(textposition="top center", textfont_size=9)
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.add_vline(x=0, line_dash="dot", line_color="gray")
    fig.update_layout(template="plotly_white", height=620,
                      font=dict(family="Space Grotesk, sans-serif"))
    return fig

plot_value_quality(results).show()

In [23]:
# ════════════════════════════════════════════════════════════════════════
# CELL 11 — VISUALIZATION 4: RADAR (factor fingerprint of one stock)
# ════════════════════════════════════════════════════════════════════════

def plot_radar(results, ticker, family_cols):
    if ticker not in results.index:
        print(f"{ticker} not in results."); return
    vals = results.loc[ticker, family_cols].tolist()
    vals += vals[:1]               # close the loop
    axes = family_cols + family_cols[:1]
    fig = go.Figure(go.Scatterpolar(
        r=vals, theta=axes, fill="toself", name=ticker,
        line_color="#1f7a5a",
    ))
    fig.update_layout(
        title=f"Factor Profile — {ticker}",
        polar=dict(radialaxis=dict(visible=True)),
        template="plotly_white", height=520,
        font=dict(family="Space Grotesk, sans-serif"),
    )
    return fig

plot_radar(results, results.index[0], fam_cols).show()

In [24]:
# ════════════════════════════════════════════════════════════════════════
# CELL 12 — PERFORMANCE METRICS & DIAGNOSTICS
# ════════════════════════════════════════════════════════════════════════
# Coverage, factor dispersion, and a simple forward-return sanity check
# (top vs bottom decile) using the most recent month already in our prices.

def coverage_report(factor_table, factor_map):
    rows = []
    for fam, metrics in factor_map.items():
        present = [m for m in metrics if m in factor_table.columns]
        cov = factor_table[present].notna().mean().mean() if present else 0
        rows.append({"family": fam, "coverage_%": round(cov * 100, 1)})
    return pd.DataFrame(rows)


def decile_spread(results, prices, skip=cfg.MOMENTUM_SKIP):
    """Mean recent-month return of the best vs worst decile (illustrative)."""
    fwd = (prices.iloc[-1] / prices.iloc[-skip] - 1.0)
    top = results[results["decile"] == 1].index
    bot = results[results["decile"] == results["decile"].max()].index
    top_r = fwd.reindex(top).mean()
    bot_r = fwd.reindex(bot).mean()
    return top_r, bot_r, top_r - bot_r


print("Factor coverage:")
print(coverage_report(factor_table, FACTOR_MAP).to_string(index=False))

t, b, spread = decile_spread(results, prices)
print(f"\nIllustrative recent-month return:")
print(f"  Top decile:    {t:+.2%}")
print(f"  Bottom decile: {b:+.2%}")
print(f"  Spread:        {spread:+.2%}")

Factor coverage:
       family  coverage_%
        value     100.000
profitability     100.000
     momentum      99.600
   volatility     100.000
       growth     100.000

Illustrative recent-month return:
  Top decile:    -2.06%
  Bottom decile: +2.73%
  Spread:        -4.79%


In [25]:
# ════════════════════════════════════════════════════════════════════════
# CELL 13 — EXPORT DELIVERABLES
# ════════════════════════════════════════════════════════════════════════
# Save the ranked leaderboard and an interactive dashboard HTML for GitHub.

def export_dashboard(results, factor_table, fam_cols, path="data/processed"):
    os.makedirs(path, exist_ok=True)
    results.to_csv(f"{path}/screener_results.csv")

    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "bar"}, {"type": "heatmap"}],
               [{"type": "scatter"}, {"type": "scatterpolar"}]],
        subplot_titles=("Top Rankings", "Factor Heatmap",
                        "Value vs Quality", "Top Pick Profile"),
    )
    top = results.head(12).iloc[::-1]
    fig.add_trace(go.Bar(x=top["composite"], y=top.index,
                         orientation="h", marker_color="#1f7a5a"), 1, 1)
    h = results.head(12)[fam_cols]
    fig.add_trace(go.Heatmap(z=h.values, x=fam_cols, y=h.index,
                             colorscale="RdYlGn", zmid=0, showscale=False), 1, 2)
    fig.add_trace(go.Scatter(x=results["value"], y=results["profitability"],
                             mode="markers", marker=dict(size=8,
                             color="#1f7a5a")), 2, 1)
    tkr = results.index[0]
    rv = results.loc[tkr, fam_cols].tolist(); rv += rv[:1]
    fig.add_trace(go.Scatterpolar(r=rv, theta=fam_cols + fam_cols[:1],
                                  fill="toself", line_color="#1f7a5a"), 2, 2)
    fig.update_layout(height=900, template="plotly_white", showlegend=False,
                      title_text="Multi-Factor Equity Screener — Dashboard")
    fig.write_html(f"{path}/dashboard.html")
    print(f"Saved leaderboard + dashboard to {path}/")

export_dashboard(results, factor_table, fam_cols)
print("\nPipeline complete.")

Saved leaderboard + dashboard to data/processed/

Pipeline complete.
